# Lineborn Sales LoRA v6
Valid JSON notebook for Qwen3-4B-Instruct-2507 sales QLoRA + DPO. Use a GPU runtime and Run all.


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

def run(*args):
    cmd=[str(x) for x in args]
    print(">", " ".join(cmd), flush=True)
    p=subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail=[]
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="", flush=True)
        tail.append(line)
        if len(tail)>220:
            tail.pop(0)
    code=p.wait()
    if code:
        raise RuntimeError("Command failed with exit code %s: %r\n\nLAST OUTPUT:\n%s" % (code, cmd, "".join(tail)))
    return code

run("nvidia-smi")
print("Notebook Python:", sys.version)
root=Path("/content/lineborn-runtime")
if root.exists():
    shutil.rmtree(root)
run("git","clone","--depth","1","--branch","lineborn-sales-lora","https://github.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime.git",str(root))
os.chdir(root)
run("git","rev-parse","HEAD")
run(sys.executable,"-m","pip","install","--upgrade","-q","jedi>=0.19.2")
run(sys.executable,"-m","pip","install","--upgrade","-q","-r","training/requirements-colab.txt")
run(sys.executable,"-c","import torch,transformers,peft,trl,datasets,accelerate,bitsandbytes; assert torch.cuda.is_available(); print('GPU',torch.cuda.get_device_name(0)); print('transformers',transformers.__version__); print('trl',trl.__version__)")


In [ ]:
import inspect
from transformers import TrainingArguments
from trl import DPOConfig, DPOTrainer
assert "warmup_steps" in inspect.signature(TrainingArguments.__init__).parameters
assert "warmup_steps" in inspect.signature(DPOConfig.__init__).parameters
run(sys.executable,"-m","py_compile","training/train_sales_lora.py","training/train_sales_dpo.py","training/package_sales_adapters.py")
sft_source=Path("training/train_sales_lora.py").read_text("utf-8")
dpo_source=Path("training/train_sales_dpo.py").read_text("utf-8")
assert "warmup_ratio=" not in sft_source
assert "warmup_ratio=" not in dpo_source
print("Training API preflight passed")


In [ ]:
run(sys.executable,"training/build_sales_corpus.py")
run(sys.executable,"training/validate_sales_corpus.py")


In [ ]:
run(sys.executable,"training/train_sales_lora.py","--max-length","1536","--epochs","2","--grad-accum","16")
sft_model=Path("training/output/lineborn-sales-sft/adapter/adapter_model.safetensors")
assert sft_model.is_file() and sft_model.stat().st_size>1024*1024
print("SFT adapter ready: %.1f MiB" % (sft_model.stat().st_size/1024/1024))


In [ ]:
run(sys.executable,"training/train_sales_dpo.py","--max-length","1536","--epochs","1","--grad-accum","16")
dpo_model=Path("training/output/lineborn-sales-dpo/adapter/adapter_model.safetensors")
assert dpo_model.is_file() and dpo_model.stat().st_size>1024*1024
print("DPO adapter ready: %.1f MiB" % (dpo_model.stat().st_size/1024/1024))


In [ ]:
archive=Path("/content/lineborn-sales-adapters-v6.zip")
if archive.exists():
    archive.unlink()
run(sys.executable,"training/package_sales_adapters.py","--require-dpo","--archive",str(archive))
assert archive.is_file() and archive.stat().st_size>2*1024*1024
print("Candidate archive ready: %.1f MiB" % (archive.stat().st_size/1024/1024))
from google.colab import files
files.download(str(archive))
